# 双均线策略入门

这个 Notebook 带你**一步一步**理解双均线策略的回测过程。

建议先阅读 `docs/01-入门指南.md`，再运行下面的代码单元。

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import yaml

from src.data.fetch_data import fetch_ohlcv
from src.strategies.dual_moving_average import dual_moving_average_signals
from src.backtest.engine import run_backtest

## 1. 加载配置并下载数据

In [ ]:
with open(ROOT / 'strategies/dual_ma/config.yaml') as f:
    config = yaml.safe_load(f)

df = fetch_ohlcv(config['symbol'], start=config['start'], end=config.get('end'))
df.tail()

## 2. 计算双均线信号

In [ ]:
signals_df = dual_moving_average_signals(
    df['Close'],
    short_window=config['short_window'],
    long_window=config['long_window'],
)
signals_df.tail(10)

## 3. 可视化：价格 + 均线

In [ ]:
plot_df = signals_df.dropna()

plt.figure(figsize=(12, 5))
plt.plot(plot_df.index, plot_df['close'], label='收盘价', alpha=0.7)
plt.plot(plot_df.index, plot_df['ma_short'], label=f"MA{config['short_window']}")
plt.plot(plot_df.index, plot_df['ma_long'], label=f"MA{config['long_window']}")
plt.title(f"{config['symbol']} 双均线")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. 运行回测并查看资金曲线

In [ ]:
result = run_backtest(signals_df['close'], signals_df['signal'], initial_cash=config['initial_cash'])

print(f"总收益率: {result.total_return * 100:.2f}%")
print(f"最大回撤: {result.max_drawdown * 100:.2f}%")
print(f"夏普比率: {result.sharpe_ratio:.2f}")

plt.figure(figsize=(12, 4))
plt.plot(result.equity_curve.index, result.equity_curve.values)
plt.title('策略资金曲线')
plt.ylabel('资金')
plt.grid(True, alpha=0.3)
plt.show()